In [1]:

import pandas as pd

import numpy as np

from ilipy import Session
from ilipy.database import DistanceCorrelation
from ilipy import TrackIndex, OdometerTicks

inspection_id = "0ABP0TFUSH1"
num_tracks = 22
environment = "prod"
model_detection_threshold = 0.99


In [2]:
df_with_final_track = pd.read_parquet(f"./insp_{inspection_id}_final_df_us_with_final_track_id.parquet")
display(df_with_final_track)

,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count,overlap_percent_with_ae,final_track_id
0,6.9,8,6.861793,7.295383,7.040086,"[16, 18]",7.146289,7.504650,7.355713,0.991347,0.9927,0.9905,53.466667,237.000000,"{'0': None, '1': None, '10': None, '11': None,...",2,0.0,16
1,7.8,3,7.791304,7.860076,7.829494,"[8, 15, 19]",7.855114,8.360874,8.032038,0.993700,0.9951,0.9927,113.333333,340.000000,"{'0': None, '1': None, '10': None, '11': None,...",3,0.0,15
2,8.1,2,8.080930,8.151168,8.116049,"[12, 13]",8.151449,8.181559,8.166504,0.993350,0.9941,0.9926,29.000000,58.000000,"{'0': None, '1': None, '10': None, '11': None,...",2,0.0,12
3,8.4,6,8.499537,8.929400,8.499537,"[1, 2, 15]",8.992399,9.651147,9.481325,0.992650,0.9933,0.9920,111.888889,192.333333,"{'0': None, '1': 0.9935, '10': None, '11': Non...",1,0.0,2
4,9.9,1,10.056602,10.056602,10.056602,[17],10.135714,10.135714,10.135714,0.992300,0.9923,0.9923,45.000000,45.000000,"{'0': None, '1': None, '10': None, '11': None,...",1,0.0,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20489,201613.2,4,201613.098983,201613.360497,201613.258646,"[3, 4, 14]",201613.293806,201613.583181,201613.439120,0.992200,0.9932,0.9912,101.250000,405.000000,"{'0': None, '1': None, '10': None, '11': None,...",3,0.0,4
20490,201613.5,1,201613.558576,201613.558576,201613.558576,[14],201613.574779,201613.574779,201613.574779,0.990400,0.9904,0.9904,10.000000,10.000000,"{'0': None, '1': None, '10': None, '11': None,...",1,0.0,14
20491,201613.8,1,201613.620981,201613.620981,201613.620981,[4],201614.078179,201614.078179,201614.078179,0.991800,0.9918,0.9918,255.000000,255.000000,"{'0': None, '1': None, '10': None, '11': None,...",1,0.0,4
20492,201615.0,2,201615.085357,201615.101602,201615.093480,"[13, 18]",201615.110066,201615.126502,201615.118284,0.991100,0.9912,0.9910,15.000000,30.000000,"{'0': None, '1': None, '10': None, '11': None,...",2,0.0,13


## Upload anomalies

In [3]:
from ilipy import ClipTypes, Session, TrackIndex, ViewDistance
from ilipy.channeldata import ImageProfile
from ilipy.features import Bookmarks
from ilipyutils.ml_features.base import (get_ml_models_info_list, AnomalyStatus)
from ilipyutils.ml_features.insert import FeatureInsert, GeometryCubeParameters
from ili_custom_data.load_model import ModelDataLoader
from ilipy.database import DistanceCorrelation
from ilipyutils.ml_features.overlap import (
    AnomalyOverlapManager,
    OverlapAction,
    OverlapPolicy,
)


session = Session(environment=environment)
session.set_active_inspection(inspection_id)
dist_corr = DistanceCorrelation(session)
bookmarks_interface = Bookmarks(connector=session.database_connector)
latest_model = ModelDataLoader.get_latest_model()

overlap_policy = OverlapPolicy(
    action=OverlapAction.TAG_DUPLICATE,
    min_overlap_iou=0.1,
)
anomaly_overlap_manager = AnomalyOverlapManager(
    overlap_policy=overlap_policy,
    session=session,
    bookmarks_interface=bookmarks_interface,
)
feature_insert = FeatureInsert(
    session=session,
    bookmarks_interface=bookmarks_interface,
    anomaly_overlap_manager=anomaly_overlap_manager, # Set to None to disable overlap checking
)

PySettings.cpp(46): ilipy: Build 0.18.0.12915 c4cd9c78a4 release-ili-0.18 'Thu Dec 18 15:02:49 2025'
PySettings.cpp(47): Paths modulePath: /home/zmirikha/ilipy/lib/python3.11/site-packages/ilipy, userHomePath: /home/zmirikha, identityPath: /home/zmirikha/.aws/ilipy
Settings.cpp(246): Settings: loading settings from /home/zmirikha/ilipy/lib/python3.11/site-packages/ilipy/Data/Environments/defaults.json
Settings.cpp(246): Settings: loading settings from /home/zmirikha/ilipy/lib/python3.11/site-packages/ilipy/Data/Environments/prod.json
PyAuthenticator.cpp(239): Attempting an IAM role login as no username and password or environment variables were provided.
CognitoAuthenticator.cpp(158): Using Cognito Credentials
CognitoAuthenticator.cpp(842): Login Successful.
CognitoAuthenticator.cpp(842): Login Successful.
CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji
CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji
DatabaseWebSocket.cpp(153): WebSocket 

In [4]:
cube_params = GeometryCubeParameters(
tlbr_cube_angle_rad=(np.deg2rad(-15), np.deg2rad(15)),
tlbr_probe_scan_angle_rad=(np.deg2rad(-15), np.deg2rad(15)),
tlbr_radial_position_mm=(200-(1.9/2), 200+(1.9/2)),
)
model_info = next(
    model
    for model in get_ml_models_info_list()
    if model.model_name == "Ultrasound-Dent-v1"
)
model_info


MLModelInfo(model_name='Ultrasound-Dent-v1', model_id=14, model_score_bound=[0, 1], anomaly_type_name='DentPlain')

In [5]:

#reset index of final_grouped_bookmarks
final_grouped_bookmarks = df_with_final_track.reset_index(drop=True)
display(final_grouped_bookmarks)

,view_distance_group,view_distance_start_count,view_distance_start_min,view_distance_start_max,view_distance_start_mean,track_ids,view_distance_stop_min,view_distance_stop_max,view_distance_stop_mean,avg_confidence_mean,avg_confidence_max,avg_confidence_min,sequence_length_mean,sequence_length_sum,max_confidence_by_track,unique_tracks_count,overlap_percent_with_ae,final_track_id
0,6.9,8,6.861793,7.295383,7.040086,"[16, 18]",7.146289,7.504650,7.355713,0.991347,0.9927,0.9905,53.466667,237.000000,"{'0': None, '1': None, '10': None, '11': None,...",2,0.0,16
1,7.8,3,7.791304,7.860076,7.829494,"[8, 15, 19]",7.855114,8.360874,8.032038,0.993700,0.9951,0.9927,113.333333,340.000000,"{'0': None, '1': None, '10': None, '11': None,...",3,0.0,15
2,8.1,2,8.080930,8.151168,8.116049,"[12, 13]",8.151449,8.181559,8.166504,0.993350,0.9941,0.9926,29.000000,58.000000,"{'0': None, '1': None, '10': None, '11': None,...",2,0.0,12
3,8.4,6,8.499537,8.929400,8.499537,"[1, 2, 15]",8.992399,9.651147,9.481325,0.992650,0.9933,0.9920,111.888889,192.333333,"{'0': None, '1': 0.9935, '10': None, '11': Non...",1,0.0,2
4,9.9,1,10.056602,10.056602,10.056602,[17],10.135714,10.135714,10.135714,0.992300,0.9923,0.9923,45.000000,45.000000,"{'0': None, '1': None, '10': None, '11': None,...",1,0.0,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20489,201613.2,4,201613.098983,201613.360497,201613.258646,"[3, 4, 14]",201613.293806,201613.583181,201613.439120,0.992200,0.9932,0.9912,101.250000,405.000000,"{'0': None, '1': None, '10': None, '11': None,...",3,0.0,4
20490,201613.5,1,201613.558576,201613.558576,201613.558576,[14],201613.574779,201613.574779,201613.574779,0.990400,0.9904,0.9904,10.000000,10.000000,"{'0': None, '1': None, '10': None, '11': None,...",1,0.0,14
20491,201613.8,1,201613.620981,201613.620981,201613.620981,[4],201614.078179,201614.078179,201614.078179,0.991800,0.9918,0.9918,255.000000,255.000000,"{'0': None, '1': None, '10': None, '11': None,...",1,0.0,4
20492,201615.0,2,201615.085357,201615.101602,201615.093480,"[13, 18]",201615.110066,201615.126502,201615.118284,0.991100,0.9912,0.9910,15.000000,30.000000,"{'0': None, '1': None, '10': None, '11': None,...",2,0.0,13


In [8]:
anomalies = bookmarks_interface.get_anomalies(inspectionId=inspection_id)
dent_anomalies = [a for a in anomalies if "Ultrasound-Dent-v1" in a.tags]
print(f"{len(dent_anomalies)} anomalies with Ultrasound-Dent-v1 tag found in {environment}.")


0 anomalies with Ultrasound-Dent-v1 tag found in prod.


/tmp/ipykernel_1748976/2572833939.py:1: DeprecationWarning: get_anomalies(inspectionId) is deprecated, use get_anomalies with pageOffset and pageSize instead.
  anomalies = bookmarks_interface.get_anomalies(inspectionId=inspection_id)


In [ ]:
#deleted_anomalies = [bookmarks_interface.delete_anomaly(a.feature.anomaly_feature_id) for a in dent_anomalies]

In [9]:

for i, row in final_grouped_bookmarks.iterrows():
        if i<15:
            continue
        profile = ImageProfile.ZeroAngle 
        vd_start, vd_end, track_id = row['view_distance_start_mean'], row['view_distance_stop_mean'], row['final_track_id']

            
        track_nums = (track_id,track_id)
        start_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(track_nums[0]), ViewDistance(vd_start)).value
        end_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(track_nums[1]), ViewDistance(vd_end)).value
        print(start_frame_odo_ticks)
        print(end_frame_odo_ticks)
        print(f"uploading row {i+1} of {len(final_grouped_bookmarks)}")
        anomaly_info = feature_insert.insert_ml_pred(
        inspection_id=inspection_id,
        tlbr_track_indices=track_nums,
        tlbr_odometer_ticks=(start_frame_odo_ticks, end_frame_odo_ticks),
        cube_params=cube_params,
        model_info=model_info,
        anomaly_status=AnomalyStatus.REVIEW_DETECTION,
        image_profile=profile,
        extra_tags=None,
        # ili_custom_data=model_instance
        )
                


280021
280903
uploading row 16 of 20494
DbHealthCheck.cpp(146): Database service health check complete: Healthy
283894
298746
uploading row 17 of 20494
302245
309697
uploading row 18 of 20494
319034
319196
uploading row 19 of 20494
322937
327607
uploading row 20 of 20494
327733
328165
uploading row 21 of 20494
336056
336399
uploading row 22 of 20494
339578
339920
uploading row 23 of 20494
342091
342163
uploading row 24 of 20494
354163
354325
uploading row 25 of 20494
372557
375754
uploading row 26 of 20494
371815
372247
uploading row 27 of 20494
389713
390055
uploading row 28 of 20494
399658
405175
uploading row 29 of 20494
418206
418459
uploading row 30 of 20494
418729
418891
uploading row 31 of 20494
424273
435235
uploading row 32 of 20494
447763
448825
uploading row 33 of 20494
476035
480991
uploading row 34 of 20494
530659
535573
uploading row 35 of 20494
543553
547135
uploading row 36 of 20494
592105
596947
uploading row 37 of 20494
647149
648301
uploading row 38 of 20494
659719
6

In [ ]:
20494-16

20478

DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: 